# Arabic OCR — prepare & vet training data (CPU)

No GPU. Assemble the sources named in a `configs/training/*.yaml`, run the
contamination check against the benchmark test splits, and eyeball a sample.
Use this before spending GPU time in `05_colab_train_lora.ipynb`.

## 1. Repo

In [ ]:
# Bulletproof checkout: works whether the repo is missing, at the wrong
# path, or has local edits. Always ends up at the latest origin/main.
import os
REPO = '/content/arabic-ocr-benchmark'
# Copy Colab secrets into the environment. SWEEP_RUNNER_ID pins a STABLE
# name for this account (e.g. 'runner-a') so a restarted runner re-adopts
# its own in-flight batches instantly instead of waiting out the lease TTL.
for _k in ('GH_TOKEN', 'HF_TOKEN', 'SWEEP_RUNNER_ID'):
    try:
        from google.colab import userdata
        _v = userdata.get(_k)
        if _v: os.environ[_k] = _v
    except Exception:
        pass
token = os.environ.get('GH_TOKEN')
if not token:
    print('no GH_TOKEN secret — clone/push will be anonymous / read-only')
import subprocess, time, sys
url = f"https://{(token + '@') if token else ''}github.com/rabeeh97/arabic-ocr-benchmark.git"
def _sh(*a):
    return subprocess.run(a, capture_output=True, text=True)
_updated = False
if os.path.isdir(REPO + '/.git'):
    r = _sh('git', '-C', REPO, 'fetch', '--depth', '1', 'origin', 'main')
    if r.returncode == 0:
        _sh('git', '-C', REPO, 'reset', '--hard', 'origin/main')
        # drop untracked leftovers (e.g. a stale sweep_manifest.json); keeps gitignored _cache
        _sh('git', '-C', REPO, 'clean', '-fd', '--', 'results')
        _updated = True
    else:
        print('existing checkout is stale and fetch failed — re-cloning\n', r.stderr[-300:])
        _sh('rm', '-rf', REPO)
if not _updated:
    for _try in range(1, 5):
        _sh('rm', '-rf', REPO)
        r = _sh('git', 'clone', '--depth', '1', url, REPO)
        if r.returncode == 0 and os.path.isdir(REPO + '/.git'):
            break
        print(f'clone attempt {_try}/4 failed:\n' + (r.stderr[-400:] or r.stdout[-400:]))
        time.sleep(5 * _try)
if not os.path.isdir(REPO + '/.git'):
    raise RuntimeError(
        'git clone failed. Checklist: (1) add a Colab secret named GH_TOKEN — a '
        'fine-grained PAT with Contents: read on rabeeh97/arabic-ocr-benchmark '
        '(the repo is private); (2) enable notebook access to that secret in the '
        'left sidebar key icon; (3) if it is a network blip, just re-run this cell.'
    )
os.chdir(REPO)
# src-layout package: importable straight from the checkout even if the
# `pip install -e` cell fails or is skipped (this is what caused the old
# 'No module named arabic_ocr_bench' in the sweep cell).
for _p in (os.path.join(os.getcwd(), 'src'), os.getcwd()):
    if _p not in sys.path: sys.path.insert(0, _p)
print(_sh('git', 'log', '--oneline', '-3').stdout)

## 2. Install — core + train extras (no torch) + Arabic fonts

In [ ]:
# Keep all downloads on Colab's own scratch disk (ephemeral, ~78 GB) --
# never the local machine. Nothing here touches your computer.
import os
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Optional HF_TOKEN secret: lifts anonymous rate limits and speeds the
# Muharaf download (and is required for gated models like PaliGemma's base).
try:
    from google.colab import userdata
    _hf = userdata.get('HF_TOKEN')
    if _hf:
        os.environ['HF_TOKEN'] = _hf
        print('HF_TOKEN set')
except Exception:
    pass

In [ ]:
!pip install -q -e '.[train]'   # datasets, imagehash, fonttools, pydantic — no torch
!python scripts/fetch_fonts.py   # extra Arabic faces for synthetic rendering (Amiri family)

## 3. Config

In [ ]:
TRAIN_CFG = 'configs/training/m1_printed_synth_qwen25vl3b_5k.yaml'
from arabic_ocr_bench.training.config import load_training_config
cfg = load_training_config(TRAIN_CFG)
print(cfg.experiment, [s.name for s in cfg.data.sources])

## 4. (synthetic sources) mount Drive + set up a resumable render cache

For a `synthetic_printed` source every line is rendered once, then reused
by `05_colab_train_lora.ipynb`. Rendering thousands of lines is slow, so
this cell makes it **resumable and Drive-friendly**:

* `work_dir` &rarr; `/content` (fast local disk) is where images are written.
* `cache_dir` &rarr; a Drive folder holds a single `images.tar` checkpoint
  plus a cached `pool_*.json` (the slow corpus + contamination step).

The cache dir is named by a **hash of the render parameters** (corpora,
seed, degradation, filters) &mdash; *not* the experiment name &mdash; so the
5k and 15k configs, and notebooks 05 and 06, all share one render cache.
On start it restores `images.tar`; it re-checkpoints every `flush_every`
new lines. A disconnect costs at most `flush_every` lines &mdash; just
re-run the assemble cell and it picks up where it stopped.

In [ ]:
import pathlib, hashlib, json
from arabic_ocr_bench.training.synth import synth_render_key
SYNTH = [s for s in cfg.data.sources if 'synth' in s.adapter.lower()]
if SYNTH:
    from google.colab import drive; drive.mount('/content/drive')
    rkey = synth_render_key(SYNTH[0].params)   # shared across configs w/ same render params
    drive_cache = pathlib.Path('/content/drive/MyDrive/arabic-ocr-bench/synth') / f'render-{rkey}'
    work = pathlib.Path('/content/synth') / f'render-{rkey}'   # fast local scratch
    drive_cache.mkdir(parents=True, exist_ok=True); work.mkdir(parents=True, exist_ok=True)
    for s in SYNTH:
        s.params['cache_dir'] = str(drive_cache)   # durable: images.tar + pool cache
        s.params['work_dir'] = str(work)           # where rendering actually happens
        s.params['flush_every'] = 200              # re-pack images.tar every 200 new lines
    print('synth render ->', work, '| checkpoint ->', drive_cache / 'images.tar')
    if (drive_cache / 'images.tar').exists():
        import tarfile
        with tarfile.open(drive_cache / 'images.tar') as _tf:
            print('  existing checkpoint:', len(_tf.getnames()), 'cached lines')

## 4b. Sanity-check one rendered line

Render + show a single line **before** the full pass. The Arabic must read
right-to-left, be joined, and keep its harakat. If it looks mirrored or
disjointed, stop and check the printed `pillow raqm` line below — a Pillow
wheel built with libraqm needs the basic-layout-engine fix in
`training/synth.py::render_line` (already applied on `main`; if you still
see this, `git pull` didn't take or the runtime needs a restart).

In [ ]:
from PIL import features
from arabic_ocr_bench.training.synth import render_line
print('pillow raqm (shaping) available:', features.check('raqm'))
_demo = 'الْعِلْمُ نُورٌ وَالْجَهْلُ ظَلَامٌ، وَالْقِرَاءَةُ غِذَاءُ الْعَقْلِ.'
display(render_line(_demo, size=40))
print('expected: reads right-to-left, letters joined, harakat visible')

## 5. Assemble + class balance by source

In [ ]:
from arabic_ocr_bench.training.data import assemble_dataset
bundle = assemble_dataset(cfg.data, seed=cfg.train.seed)   # renders synthetic lines (slow first time)
from collections import Counter
print('train', len(bundle['train']), 'val', len(bundle['val']))
print(Counter(r['source'] for r in bundle['train'] + bundle['val']))

## 6. Contamination report

In [ ]:
from arabic_ocr_bench.training.contamination import check_datasets
specs = [{**s.model_dump(exclude_none=True), 'name': s.name} for s in cfg.data.sources]
rep = check_datasets(specs, cfg.eval_datasets, limit=None)
import json; print(json.dumps(rep['pairs'], ensure_ascii=False, indent=2)[:4000])
print('flagged pairs:', rep['n_flagged'])

## 7. Eyeball samples

In [ ]:
import random
for r in random.Random(0).sample(bundle['train'], 8):
    display(r['images'][0])
    print(r['source'], '|', r['messages'][2]['content'][0]['text'])

## 8. Commit the contamination report

It's cited in every experiment log and the writeup — keep it in git.

In [ ]:
import subprocess
if token:
    subprocess.run(['git','config','user.email','colab@arabic-ocr-bench'], check=True)
    subprocess.run(['git','config','user.name','colab-prep'], check=True)
    subprocess.run(['git','add','-f','results/contamination_report.json'], check=True)
    r = subprocess.run(['git','commit','-m',f'contam: {cfg.experiment}'], capture_output=True, text=True)
    print(r.stdout or r.stderr)
    push = subprocess.run(['git','pull','--rebase','--autostash','origin','main'], capture_output=True, text=True)
    print(push.stdout or push.stderr)
    push = subprocess.run(['git','push','origin','HEAD:main'], capture_output=True, text=True)
    print(push.stdout or push.stderr)
else:
    print('no GH_TOKEN -- contamination_report.json is local only; download it from the file browser')